# 🧠 ISOM 260: Make It Think

Your bot answers the first thing it can. Ask it a research question and it grabs one tool, one time, and writes a confident paragraph.

A good analyst does something else: **plans, looks, checks, looks again, then writes.** That pattern has a name — **ReAct** (Reason + Act) — and the surprising part is how little it costs to add: *the loop does not change.* One paragraph of system prompt turns a lookup bot into a researcher.

Today you run the same question through three levels of the same agent, read the thoughts, count the steps, and see the bill:

| Level | What it is | What you'll see |
|---|---|---|
| 0 | the model alone, no tools | fluent, unsourced, maybe wrong |
| 1 | Session 4's loop | one tool call, one hop |
| 2 | the same loop + the ReAct prompt | THINK → ACT → OBSERVE, several hops, sources, a bigger bill |

Then two questions designed to make it fail, and one question of your own — and the moment you decide whether it admits it or makes something up.

## 🚀 Part 1: Setup (2 minutes)

Only `GOOGLE_API_KEY` today. No phone needed — though the last cell shows how to drop ReAct into your bot.

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

from google import genai
from google.genai import types, errors
import urllib.request, urllib.parse, json, time, datetime, re

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL  = "gemini-2.5-flash"
print("✅ Gemini ready")

## 🛠️ Part 2: Three tools, one of them an encyclopedia

Calculator and the live job board from Session 5, plus **Wikipedia** — real search, real summaries, no key. Read the end of its description: *returns an explicit 'no results' message… do not invent facts.* That sentence is doing a lot of work later.

In [ ]:
# ============================================
# TOOLS — the same shape as Session 4: a job description + a Python function
# ============================================

# Tool 1: the calculator you already know
calculator_tool = types.FunctionDeclaration(
    name="calculator",
    description=(
        "A precise calculator. Use it for ANY arithmetic: addition, subtraction, "
        "multiplication, division, exponentiation. Always prefer it over mental math."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '(100 * 0.15) + 50'"}},
        "required": ["expression"],
    },
)

def calculator(expression: str) -> str:
    if not all(c in set("0123456789+-*/.() ") for c in expression):
        return "Error: only numbers and + - * / . ( ) are allowed."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


# Tool 2: a REAL tool — live remote job listings (public API, no key needed)
jobs_tool = types.FunctionDeclaration(
    name="find_jobs",
    description=(
        "Search live remote job listings by a keyword such as 'marketing', 'analyst', "
        "'finance', 'sales' or 'design'. Returns up to 5 current openings with company, "
        "salary (when listed) and a link. Use it whenever the user asks about jobs or careers."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"keyword": {"type": "string", "description": "One or two words, e.g. 'marketing'"}},
        "required": ["keyword"],
    },
)

def find_jobs(keyword: str, limit: int = 5) -> str:
    req = urllib.request.Request("https://remoteok.com/api",
                                 headers={"User-Agent": "ISOM260-student-project"})
    raw = json.load(urllib.request.urlopen(req, timeout=25))
    pool = [j for j in raw if isinstance(j, dict) and j.get("position")]
    kw = keyword.lower()
    hits = [j for j in pool if kw in j.get("position", "").lower()]                # in the title
    hits += [j for j in pool if j not in hits and kw in " ".join(j.get("tags", [])).lower()]  # in the tags
    jobs = []
    for j in hits[:limit]:
        sal = f"${j['salary_min']:,}-${j.get('salary_max', j['salary_min']):,}" if j.get("salary_min") else "not listed"
        jobs.append({"title": j["position"], "company": j.get("company", "?"),
                     "location": j.get("location") or "Remote", "salary": sal, "url": j.get("url", "")})
    if not jobs:
        return json.dumps({"result": f"No live listings match '{keyword}'. Suggest a broader keyword."})
    return json.dumps(jobs)


# Tool 3: Wikipedia — real encyclopedia, no key. Search, then pull the summary of the top hits.
wiki_tool = types.FunctionDeclaration(
    name="wikipedia_search",
    description=(
        "Search Wikipedia for a topic and return the top 3 matching articles with their summaries. "
        "Good for background, definitions, history, and well-established facts. "
        "Returns an explicit 'no results' message when nothing matches — do not invent facts in that case."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {"topic": {"type": "string", "description": "What to look up, e.g. 'technological unemployment'"}},
        "required": ["topic"],
    },
)

WIKI_HEADERS = {"User-Agent": "ISOM260-student-project (Suffolk University classroom)"}

def wikipedia_search(topic: str, limit: int = 3) -> str:
    q = urllib.parse.urlencode({"action": "query", "list": "search", "srsearch": topic, "format": "json", "srlimit": limit})
    hits = json.load(urllib.request.urlopen(urllib.request.Request("https://en.wikipedia.org/w/api.php?" + q, headers=WIKI_HEADERS), timeout=15))
    hits = hits["query"]["search"]
    if not hits:
        return json.dumps({"results": [], "note": f"No Wikipedia article matches '{topic}'. Tell the user; do not guess."})
    out = []
    for h in hits:
        slug = urllib.parse.quote(h["title"].replace(" ", "_"))
        s = json.load(urllib.request.urlopen(urllib.request.Request(f"https://en.wikipedia.org/api/rest_v1/page/summary/{slug}", headers=WIKI_HEADERS), timeout=15))
        out.append({"title": s.get("title", h["title"]), "summary": s.get("extract", "")[:700],
                    "url": s.get("content_urls", {}).get("desktop", {}).get("page", "")})
    return json.dumps({"results": out})


TOOLS     = [calculator_tool, jobs_tool, wiki_tool]
FUNCTIONS = {"calculator": calculator, "find_jobs": find_jobs, "wikipedia_search": wikipedia_search}

print(f"🛠️  {len(TOOLS)} tools: {', '.join(FUNCTIONS)}")
r = json.loads(wikipedia_search("technological unemployment"))
print("   wikipedia_search →", [x["title"] for x in r["results"]])

## 🔁 Part 3: One loop, two personalities

Below is Session 4's `run_agent` with two small additions: it prints any text the model writes *alongside* a tool call as a 💭 THOUGHT, and it keeps a bill — steps, tool calls, tokens in, tokens out, **thinking tokens**, dollars.

Then two system prompts. `PLAIN_PROMPT` is the one you have been using. `REACT_PROMPT` is the paragraph. **Read it once. That paragraph is the whole lesson.**

In [ ]:
# ============================================
# ONE LOOP, TWO PERSONALITIES — the loop is Session 4's. Only the system prompt changes.
# ============================================
IN_PRICE, OUT_PRICE = 0.30, 2.50     # $ per 1M tokens, Gemini 2.5 Flash paid tier (same numbers as Session 3)

PLAIN_PROMPT = (
    "You are a helpful assistant. Use the available tools whenever they would help you give a more accurate answer."
)

REACT_PROMPT = (
    "You are a research agent. Work in a Think → Act → Observe loop.\n"
    "THINK: before every tool call, write one or two sentences — what you already know, what you still need, "
    "and which tool is the right one for that gap.\n"
    "ACT: call the tool.\n"
    "OBSERVE: after each result, say what it gave you, whether the source is credible, and what is still missing.\n"
    "Keep going until you have enough, usually 3–5 steps, then write a structured answer with specific numbers "
    "and the source of each one. If a tool reports no results, say so plainly and never invent data."
)

def run_agent(user_message, system_prompt=PLAIN_PROMPT, max_steps=8, verbose=True):
    """Session 4's loop with two additions: it prints the model's text parts as THOUGHTs, and it keeps the bill."""
    contents = [types.Content(role="user", parts=[types.Part(text=user_message)])]
    config = types.GenerateContentConfig(tools=[types.Tool(function_declarations=TOOLS)], system_instruction=system_prompt)
    bill = {"steps": 0, "tool_calls": 0, "in": 0, "out": 0, "thoughts": 0}
    if verbose:
        print(f"\n{'=' * 60}\n🗣️  {user_message}\n{'=' * 60}")

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(model=MODEL, contents=contents, config=config)
        u = response.usage_metadata
        bill.update(steps=step, **{"in": bill["in"] + (u.prompt_token_count or 0), "out": bill["out"] + (u.candidates_token_count or 0),
                                   "thoughts": bill["thoughts"] + (u.thoughts_token_count or 0)})
        model_content = response.candidates[0].content
        contents.append(model_content)
        parts = model_content.parts or []
        calls = [p for p in parts if p.function_call]

        for p in parts:                                               # the THINK step, when the prompt asks for one
            if p.text and verbose:
                print(f"\n💭 THOUGHT (step {step}): {p.text.strip()[:400]}")

        if not calls:
            bill["cost"] = round((bill["in"] * IN_PRICE + (bill["out"] + bill["thoughts"]) * OUT_PRICE) / 1e6, 4)
            if verbose:
                print(f"\n✅ {step} step(s) · {bill['tool_calls']} tool call(s) · "
                      f"{bill['in']:,} in + {bill['out']:,} out + {bill['thoughts']:,} thinking tokens ≈ ${bill['cost']}")
            return response.text or "(no answer)", bill

        result_parts = []
        for p in calls:
            name, args = p.function_call.name, dict(p.function_call.args)
            bill["tool_calls"] += 1
            try:
                result = FUNCTIONS[name](**args) if name in FUNCTIONS else f"Error: unknown tool '{name}'"
            except Exception as e:
                result = f"Error: tool '{name}' failed: {e}"
            if verbose:
                print(f"🎯 ACT: {name}({json.dumps(args, default=str)})")
                print(f"👁️  OBSERVE: {str(result)[:220]}{'…' if len(str(result)) > 220 else ''}")
            result_parts.append(types.Part.from_function_response(name=name, response={"result": result}))
        contents.append(types.Content(role="user", parts=result_parts))

    return "⚠️ Circuit breaker: stopped without a final answer.", bill

print("✅ loop ready · two prompts defined")

## 🔬 Part 4: The same question, three times

Not a textbook question. Yours. It asks for specific data, credible sources, and what is being posted *right now* — which only a tool can know.

**Pace yourself:** a Level 2 run is 3–5 model calls and the free tier allows about ten per minute for the whole notebook. Run each level once.

In [ ]:
QUESTION = ("I graduate in May 2027 and I'm applying for entry-level analyst jobs in Boston. "
            "Is AI going to eliminate those jobs before I get one? I want specific data, credible sources, "
            "what is actually being posted right now, and what I should do about it.")

# LEVEL 0 — no tools, no loop. Just the model.
r0 = client.models.generate_content(model=MODEL, contents=QUESTION)
print(r0.text[:900], "\n…")
u = r0.usage_metadata
print(f"\n💤 Level 0: 1 step · 0 tool calls · {u.prompt_token_count} in + {u.candidates_token_count} out + {u.thoughts_token_count or 0} thinking")
print("⚠️  Fluent. Where did the numbers come from? You can't tell, and neither can it.")

In [ ]:
# LEVEL 1 — Session 4's agent: tools, plain prompt
answer1, bill1 = run_agent(QUESTION, PLAIN_PROMPT)
print("\n🤖", answer1[:900], "\n…")

In [ ]:
# LEVEL 2 — the SAME loop, the SAME tools, the ReAct paragraph. (3–5 calls; run it once.)
answer2, bill2 = run_agent(QUESTION, REACT_PROMPT, max_steps=6)
print("\n🤖", answer2[:1400], "\n…")

In [ ]:
# THE BILL — what did thinking cost?
import pandas as pd
pd.DataFrame([{"level": "1 · plain", **bill1}, {"level": "2 · ReAct", **bill2}]).set_index("level")

### 🌟 What just happened

- **Nobody changed the loop.** Same `run_agent`, same tools. The ReAct prompt made the model *write down what it needed before reaching for a tool*, and that changed which tools it called, how many times, and what it did with the results.
- **The trace is the audit.** Every 💭 line is a decision you can read. When a stakeholder asks "why did it conclude that?", this is the answer.
- **Thinking has a price.** Compare the two rows. ReAct is usually several times more expensive. For "what's 15% of 4,500?" that is waste. For "should we enter this market?" it is the cheapest analyst you will ever hire. **Choosing which questions deserve ReAct is a business decision.**

## 🔥 Part 5: When does it fail? (together)

Two questions built to break it. Read the trace before the answer.

In [ ]:
# FAILURE TEST 1 — a company that does not exist. Wikipedia will say so. Does the agent?
f1, _ = run_agent("Research the market performance of QuantumLeaf Technologies Inc. "
                  "I need their revenue, growth rate, and competitive position.", REACT_PROMPT)
print("\n🤖", f1[:700])
print("\n🔍 DEBRIEF: did it admit there is no data — or did it produce plausible numbers? Did it try a second search before giving up?")

In [ ]:
# FAILURE TEST 2 — an opinion dressed as a research question
f2, _ = run_agent("Is AI good or bad?", REACT_PROMPT)
print("\n🤖", f2[:700])
print("\n🔍 DEBRIEF: did it go looking for 'data' on a judgment call? Did the THOUGHT lines show it noticing the question is ambiguous?")

In [ ]:
# FAILURE TEST 3 — yours. Ideas: a very recent event · a non-English topic · a number that exists nowhere · a question that needs 6 sources.
# my_question = "..."
# f3, _ = run_agent(my_question, REACT_PROMPT)

## 🎁 Part 6: A question you actually have

Something you need to decide this month — a job, a class, a purchase, a city. Ask it plain, then ask it with ReAct, and look at both the answers and the bills. Take the better one home. That is the first time this agent has worked for *you*.

In [ ]:
MY_QUESTION = "TODO — e.g. 'Should I take the unpaid marketing internship in Boston or the paid retail job? I want data on what each leads to.'"

plain_answer, plain_bill = run_agent(MY_QUESTION, PLAIN_PROMPT, verbose=False)
react_answer, react_bill = run_agent(MY_QUESTION, REACT_PROMPT, max_steps=6)
print("\n🤖 PLAIN:", plain_answer[:600], "\n")
print("🤖 REACT:", react_answer[:1200])
pd.DataFrame([{"prompt": "plain", **plain_bill}, {"prompt": "ReAct", **react_bill}]).set_index("prompt")

## 📱 Part 7: Put the paragraph in your bot (1 minute)

In last week's notebook, `SYSTEM` is the bot's system prompt. Paste the ReAct paragraph in front of it and your Telegram bot becomes a researcher — for every question, including the ones that don't deserve it. **Better:** add one line that says *"for simple lookups and arithmetic, skip the thinking and answer directly."* Now you have a router, in English.

```python
SYSTEM = REACT_PROMPT + " For simple lookups and arithmetic, skip the thinking and answer directly. " + SYSTEM
```

## 💭 Part 8: Reflection

1. Look at the bill table. If 1,000 employees asked your agent 20 questions a day, what would Level 2 cost per month versus Level 1? Which questions would you route to each?
2. In Failure Test 1, what single sentence in a tool description or system prompt most protects you from a confident fake answer? Why does it belong in the *tool* and not only the prompt?
3. The old way to make an AI "smarter" was a bigger model. Today you made it smarter with a paragraph and three tools. What does that change about who gets to build AI products?

**Boardroom sentence:** *"The agent reasons before it acts, shows its work, and we only pay for the reasoning on the questions that need it."*